# Reasoning SLM - Stage 3: GRPO

## Introduction

Stage 3 starts from the Stage 2 SFT model and trains it with GRPO on the same GSM8K train problems. Nothing imitates a reference solution: the model samples several answers per problem, and the answers the checker marks correct are reinforced.

- Starting point: the merged SFT model, with a fresh LoRA adapter for GRPO.
- Rewards: correctness (1/0, the same cut and extractor as eval) plus a small format bonus (0.1) for a final `#### <number>` line.
- A short smoke run (100 steps) first, to check that the reward rises before a long run.
- The same eval as Stages 1 and 2: the 499-problem validation subset, greedy decoding, at most 512 new tokens, the same stop rules, and the 20-prompt general check.

**Target from Stage 2:** SFT fixed the format but stayed at 46.5%, while Qwen's instruct model solves about 61.5% with the format fixed. Can GRPO move the model from about 47% toward about 61%?

## Setup

Clone the repository and install the requirements, then install vLLM, which GRPO uses to sample completions. vLLM is installed here only, not in `requirements.txt`: it pins its own torch version, while Stages 1 and 2 run on Colab's default one. TRL 1.16 supports vLLM 0.22 to 0.31. Restart the runtime once after the install, then run the notebook from the top.

The SFT model, checkpoints, the adapter and the merged model live on Google Drive (`/content/drive/MyDrive/slm-post-training`). Results are written to `results/stage3/`. Download them after each run, because `/content` is wiped when the runtime disconnects.

In [1]:
import os

REPO_URL = "https://github.com/JuaniLlaberia/slm-post-training.git"
REPO_DIR = "/content/slm-post-training"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

/content/slm-post-training


In [ ]:
!pip install -q "vllm>=0.22,<0.32"

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


### Imports

In [3]:
from src.config import RESULTS_DIR, SEED
from src.data.gsm8k import load_gsm8k
from src.evaluation.general_check import GENERAL
from src.evaluation.generation import generate
from src.evaluation.harness import evaluate, save_results
from src.prompts.few_shot import STOP, build_prompt
from src.training.merge import merge_adapter
from src.training.rewards import correctness_reward, format_reward

from peft import LoraConfig
from transformers import AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

DRIVE_DIR = "/content/drive/MyDrive/slm-post-training"
SFT_MODEL = f"{DRIVE_DIR}/models/sft-merged"
STAGE_DIR = RESULTS_DIR / "stage3"

W1024 18:11:24.414000 3041 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1024 18:11:24.461000 3041 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


## Data

The same train split as Stages 1 and 2 (6,974 problems), with a different shape. GRPO needs only the prompt and the gold answer, no solution, since the model generates its own answers. The prompt is the zero-shot `Question: … Answer:` prompt used for SFT. Reusing the questions SFT already saw is standard: in RL the model learns from its own sampled answers, not from the reference solutions.

The validation subset stays out of training and is only used to evaluate afterwards.

In [4]:
train_dataset, val_dataset, test_dataset = load_gsm8k()

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Train dataset size: 6974
Validation dataset size: 499
Test dataset size: 1319


In [5]:
def to_grpo(ex):
    """
    Turns a GSM8K example into a GRPO row: the zero-shot prompt and the gold answer, no solution.
    """
    return {"prompt": build_prompt(ex["question"]),
            "gold": ex["gold"]}

grpo_train = train_dataset.map(to_grpo, remove_columns=train_dataset.column_names)
grpo_train

Map:   0%|          | 0/6974 [00:00<?, ? examples/s]

Dataset({
    features: ['gold', 'prompt'],
    num_rows: 6974
})

## Rewards

`correctness_reward` gives 1.0 when the extracted answer equals the gold, else 0.0. GRPO sampling only stops at EOS or the token limit, so the reward first cuts each completion at `\nQuestion:`, as `generate()` does at eval time, then applies the same `answer_extractor`: the RL checker and the eval checker agree. `format_reward` adds 0.1 when the completion ends with a `#### <number>` line. Both live in `src/training/rewards.py`, and TRL passes the dataset's `gold` column to them.

As a check, scoring the saved Stage 2 outputs with the rewards should reproduce the Stage 2 eval exactly.

In [6]:
import json

with open(RESULTS_DIR / "stage2" / "sft_0shot_val500.json", encoding="utf-8") as file:
    sft_rows = json.load(file)

outputs = [r["output"] for r in sft_rows]
correct = correctness_reward(outputs, [r["gold"] for r in sft_rows])
formatted = format_reward(outputs)

print(f"correctness: {sum(correct) / len(correct):.1%}  (Stage 2 eval: 46.5%)")
print(f"format: {sum(f > 0 for f in formatted) / len(formatted):.1%}")

correctness: 46.5%  (Stage 2 eval: 46.5%)
format: 98.4%


The rewards reproduce the Stage 2 eval (46.5%). 98.4% of the SFT outputs already end with a `####` line, so the format bonus is close to constant: it guards the format rather than teaching it.

## Model

The merged SFT model is the starting point, and GRPO trains a fresh LoRA adapter on top: rank 16, alpha 32, every linear layer, as in Stage 2, but without dropout, which would add noise to the log-probabilities GRPO compares. With `beta > 0`, the reference model for the KL penalty is the same model with the adapter disabled (the SFT model), so it costs no extra memory.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL)

RANK = 16
ALPHA = 32

lora_config = LoraConfig(
    r=RANK,
    lora_alpha=ALPHA,
    target_modules="all-linear",
    lora_dropout=0.0,
    bias="none",
)

## Training

GRPO starts with a 100-step smoke run, to check that training is stable and the reward rises, then the full run. The config below is the full run; the smoke run's results are in `results/stage3/grpo_smoke_*` (its logs in `grpo_smoke_logs.csv`).

The smoke run (learning rate 5e-6, `beta=0`, otherwise the same config) was stable: no length blow-up, flat entropy, no `grad_norm` spikes. Its sampled reward rose from about 25% to 34%, but greedy validation accuracy didn't move: 46.1% at step 50 and 48.7% at step 100, against SFT's 46.5%. The general check was unchanged (8/20 content, 3/20 clean). The rising reward was the model sharpening its sampling toward answers it could already give greedily, and 100 steps at 5e-6 were too few to change more.

Each step samples 8 completions for each of 8 prompts (16 completions per batch × 4 accumulation steps) at temperature 0.8: GRPO needs varied answers to compare, so sampling is not greedy. Completions are limited to 512 tokens, as in eval, and vLLM generates them on the same GPU as training. The learning rate is much lower than SFT, because RL is easy to destabilize.

The full run changes only what the smoke run pointed to, so its result can be traced to those changes:

| Parameter | Smoke run | Full run | Why |
| --- | --- | --- | --- |
| `learning_rate` | 5e-6 | 1e-5 | Training was stable, but accuracy didn't move. LoRA usually needs about 10x the full fine-tuning rate (TRL's default is 1e-6) |
| `max_steps` | 100 | 800 | About 0.9 epoch at 8 prompts per step, so no problem is seen twice |
| `save_steps` | 50 | 100 | 8 checkpoints to choose from on validation |
| `beta` | 0.0 | 0.001 | Barely constrains training, but makes TRL log `kl`: how far GRPO moves from SFT |
| `logging_steps` | 5 | 10 | Each logged row averages 80 prompts instead of 40 |

Unchanged: `num_generations=8`, `temperature=0.8`, `loss_type="dapo"` (TRL 1.16 default, no length bias), `scale_rewards="group"` (TRL 1.16 default) and both rewards.

In [ ]:
import sys

# Jupyter's stdout/stderr have no OS file handle, which colocated vLLM asks for at startup
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

In [ ]:
RUN_NAME = "grpo_full"

args = GRPOConfig(
    output_dir=f"{DRIVE_DIR}/grpo/{RUN_NAME}",
    max_steps=800,
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    num_generations=8,
    max_completion_length=512,
    temperature=0.8,
    beta=0.001,
    loss_type="dapo",
    scale_rewards="group",
    use_vllm=True,
    vllm_mode="colocate",
    vllm_max_model_length=1024,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    disable_tqdm=True,
    seed=SEED,
    report_to="none",
)

trainer = GRPOTrainer(
    model=SFT_MODEL,
    reward_funcs=[correctness_reward, format_reward],
    args=args,
    train_dataset=grpo_train,
    peft_config=lora_config,
    processing_class=tokenizer,
)

print({k: getattr(args, k) for k in ["beta", "num_generations", "loss_type", "scale_rewards",
                                     "learning_rate", "temperature", "max_steps"]})

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Capturing CUDA graphs (FULL): 100%|██████████| 11/11 [00:00<00:00, 29.74it/s]


{'beta': 0.001, 'num_generations': 8, 'loss_type': 'dapo', 'scale_rewards': 'group', 'learning_rate': 1e-05, 'temperature': 0.8, 'max_steps': 800}


In [10]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


{'loss': '0.05264', 'grad_norm': '0.1917', 'learning_rate': '9.888e-06', 'num_tokens': '1.116e+05', 'completions/mean_length': '106.7', 'completions/min_length': '33.4', 'completions/max_length': '334.7', 'completions/clipped_ratio': '0.001563', 'completions/mean_terminated_length': '106.1', 'completions/min_terminated_length': '33.4', 'completions/max_terminated_length': '326.8', 'rewards/correctness_reward/mean': '0.25', 'rewards/correctness_reward/std': '0.4261', 'rewards/format_reward/mean': '0.09938', 'rewards/format_reward/std': '0.005', 'reward': '0.3494', 'reward_std': '0.4265', 'frac_reward_zero_std': '0.3375', 'sampling/sampling_logp_difference/mean': '0.01435', 'sampling/sampling_logp_difference/max': '0.519', 'sampling/importance_sampling_ratio/min': '0.2464', 'sampling/importance_sampling_ratio/mean': '0.9757', 'sampling/importance_sampling_ratio/max': '2.13', 'kl': '0.0005967', 'entropy': '0.3932', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/regio

TrainOutput(global_step=800, training_loss=-0.01241018323926255, metrics={'train_runtime': 2924.3239, 'train_samples_per_second': 17.508, 'train_steps_per_second': 0.274, 'train_loss': -0.01241018323926255, 'epoch': 0.9184845005740528})

In [11]:
import pandas as pd

logs = pd.DataFrame(trainer.state.log_history)
cols = ["step", "rewards/correctness_reward/mean", "rewards/format_reward/mean", "frac_reward_zero_std",
        "completions/mean_length", "completions/clipped_ratio", "entropy", "grad_norm", "kl"]
print(logs[[c for c in cols if c in logs]].dropna(subset=["rewards/correctness_reward/mean"]).to_string(index=False))

STAGE_DIR.mkdir(parents=True, exist_ok=True)
logs.to_csv(STAGE_DIR / f"{RUN_NAME}_logs.csv", index=False)

 step  rewards/correctness_reward/mean  rewards/format_reward/mean  frac_reward_zero_std  completions/mean_length  completions/clipped_ratio  entropy  grad_norm       kl
   10                         0.250000                    0.099375                0.3375               106.700000                   0.001563 0.393202   0.191675 0.000597
   20                         0.320312                    0.098906                0.3750                97.443750                   0.004687 0.384861   0.393616 0.000761
   30                         0.300000                    0.099688                0.3125               101.912500                   0.000000 0.366121   0.503489 0.000843
   40                         0.370312                    0.099219                0.2375                91.450000                   0.000000 0.369749   0.261590 0.001217
   50                         0.345313                    0.099375                0.2500                96.506250                   0.000000 0.381666 

Averaged over 100-step blocks:

| Steps | Reward | Zero-std groups | Length (tokens) | Entropy | `kl` |
| --- | --- | --- | --- | --- | --- |
| 10-100 | 0.344 | 0.29 | 97 | 0.363 | 0.002 |
| 110-200 | 0.423 | 0.29 | 97 | 0.304 | 0.013 |
| 210-300 | 0.492 | 0.30 | 103 | 0.228 | 0.052 |
| 310-400 | 0.550 | 0.36 | 113 | 0.168 | 0.118 |
| 410-500 | 0.566 | 0.36 | 118 | 0.153 | 0.144 |
| 510-600 | 0.593 | 0.35 | 117 | 0.146 | 0.181 |
| 610-700 | 0.609 | 0.40 | 114 | 0.142 | 0.183 |
| 710-800 | 0.593 | 0.41 | 119 | 0.144 | 0.211 |

The sampled reward rises from 34% to about 60% and plateaus from about step 500, while `kl` keeps growing: late in the run the model keeps moving away from SFT for little reward. Entropy falls from 0.36 to 0.14, mostly by step 400, then flattens rather than collapsing. `frac_reward_zero_std` rises from 0.29 to 0.41 as more groups come out all correct, so fewer prompts carry a learning signal, which fits the plateau. Completions get about 23% longer while the truncated share stays near 0.1%, and `grad_norm` stays between 0.36 and 0.50 on average. At `beta=0.001`, the KL penalty is about 0.0002: a measurement, not a constraint.

Unlike the smoke run, the final sampled reward (about 60% at temperature 0.8) is above SFT's greedy 46.5%, a first sign of a real gain. These are train problems, though, which SFT also saw with their solutions, so only validation can tell.

## GRPO Model Eval

Training checkpoints hold only the adapter. `merge_adapter` merges each one into the SFT model, and the harness scores it zero-shot on the validation subset kept out of training, with the Stage 1 settings. Every number comes from this same merge path: in the smoke run, the same step-100 weights scored 48.7% through `merge_adapter` and 47.3% through `trainer.model.merge_and_unload()` (that second eval wasn't kept), so merge rounding alone moved the greedy accuracy by 1.4 points.

Each checkpoint is also compared with SFT problem by problem, on the same 499 problems: `only_grpo` and `only_sft` count the problems only one of them solves. `z` = (only_grpo − only_sft) / √(only_grpo + only_sft) is McNemar's paired test: if the two models were equally good, each disagreement would be a coin flip, and |z| > 2 means a gap this large would happen by chance less than 5% of the time.

In [12]:
import glob

questions = val_dataset["question"]
golds = val_dataset["gold"]

summary = []
for ckpt in sorted(glob.glob(f"{args.output_dir}/checkpoint-*"), key=lambda p: int(p.split("-")[-1])):
    step = int(ckpt.split("-")[-1])
    path = merge_adapter(SFT_MODEL, ckpt, "/content/checkpoint-merged")
    rows = evaluate(path, lambda q, _: build_prompt(q), questions, golds, stop=STOP)
    save_results(rows, STAGE_DIR / f"{RUN_NAME}_step{step}_0shot_val500.json")

    only_grpo = sum(r["correct"] and not s["correct"] for r, s in zip(rows, sft_rows))
    only_sft = sum(s["correct"] and not r["correct"] for r, s in zip(rows, sft_rows))
    summary.append({"step": step,
                    "acc": sum(r["correct"] for r in rows) / len(rows),
                    "only_grpo": only_grpo,
                    "only_sft": only_sft,
                    "z": (only_grpo - only_sft) / max(1, only_grpo + only_sft) ** 0.5})

print(pd.DataFrame(summary).to_string(index=False))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/content/checkpoint-merged: acc=47.9%  no-answer=1.0%  hit-limit=1.0%


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

/content/checkpoint-merged: acc=64.9%  no-answer=0.2%  hit-limit=0.2%


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/content/checkpoint-merged: acc=66.7%  no-answer=0.4%  hit-limit=0.4%
 step      acc  only_grpo  only_sft        z
  100 0.478958         36        29 0.868243
  200 0.551102         74        31 4.196370
  300 0.589178         94        32 5.523399
  400 0.647295        125        34 7.216769
  500 0.651303        128        35 7.284322
  600 0.665331        131        31 7.856742
  700 0.649299        126        34 7.273239
  800 0.667335        133        32 7.862839


Accuracy climbs from 47.9% at step 100 (not yet distinguishable from SFT, like the smoke run) to 64.7% at step 400, then plateaus at 64.7-66.7% through step 800. Every checkpoint from step 200 on is clearly above SFT (`z` from 4.2 to 7.9): GRPO solves about 130 problems SFT misses and loses about 32 that SFT solved, a number that stays flat across checkpoints. Validation never drops, so there is no sign of overfitting. From step 300 on, the no-answer rate is 0-0.4% (SFT: 1.2%), and answers get longer than SFT's (112 against 93 tokens on average at step 400).

### Checkpoint Choice

Differences of 1-2 points between plateau checkpoints are within noise. The rule, set before looking at the results: take the earliest checkpoint on the plateau, unless a later one is clearly better problem by problem. Step 400 is compared with each later checkpoint (`only_new` and `only_400`, then `z`).

In [13]:
def load(step):
    with open(STAGE_DIR / f"{RUN_NAME}_step{step}_0shot_val500.json", encoding="utf-8") as file:
        return json.load(file)

ref = load(400)
for step in [500, 600, 700, 800]:
    rows = load(step)
    only_new = sum(r["correct"] and not s["correct"] for r, s in zip(rows, ref))
    only_400 = sum(s["correct"] and not r["correct"] for r, s in zip(rows, ref))
    print(step, only_new, only_400, round((only_new - only_400) / max(1, only_new + only_400) ** 0.5, 2))

500 24 22 0.29
600 27 18 1.34
700 32 31 0.13
800 32 22 1.36


No later checkpoint is clearly better than step 400 (`z` at most 1.36). They swap borderline problems back and forth rather than adding new ones: step 800 solves 32 problems step 400 misses and loses 22. Step 400 is the choice: on the plateau, with the least drift from SFT (`kl` about 0.13, against about 0.21 in the last 100 steps). It was still chosen on validation, so its 64.7% is slightly optimistic, though less so than the 66.7% of the best of 8 checkpoints.

The chosen checkpoint is merged to Drive, through the same `merge_adapter` path it was scored with: this is the model for the general check. Its adapter is copied next to the Stage 2 one.

In [14]:
import shutil

BEST = 400

ckpt = f"{args.output_dir}/checkpoint-{BEST}"
GRPO_MODEL = merge_adapter(SFT_MODEL, ckpt, f"{DRIVE_DIR}/models/{RUN_NAME}-step{BEST}-merged")

adapter_dir = f"{DRIVE_DIR}/adapters/{RUN_NAME}-step{BEST}"
os.makedirs(adapter_dir, exist_ok=True)
for name in ["adapter_config.json", "adapter_model.safetensors"]:
    shutil.copy(f"{ckpt}/{name}", adapter_dir)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## General Check

The same 20 prompts and pass rules as Stages 1 and 2 (content and clean), on the chosen checkpoint, with the zero-shot `Question: … Answer:` prompt.

In [15]:
general_results = generate(GRPO_MODEL, lambda q, _: build_prompt(q), [g["prompt"] for g in GENERAL], stop=STOP, max_new_tokens=256)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [16]:
save_results(general_results, STAGE_DIR / f"{RUN_NAME}_step{BEST}_general.json")

In [17]:
general_results

[{'output': " Dear [Recipient's Name], I am writing to decline your invitation to attend our upcoming meeting. I am currently working on a project that requires me to be available for a few days, and I am not able to commit to attending the meeting. I appreciate your consideration and look forward to hearing from you soon. Thank you for your time and consideration.",
  'n_tokens': 70,
  'hit_limit': False},
 {'output': " Dear [Friend's Name],\n\nI hope this message finds you well. I wanted to take a moment to express my gratitude for all the hard work and support you provided during my move. Your help was invaluable, and I can't thank you enough for everything you did.\n\nFirst and foremost, I want to express my deepest appreciation for the time and effort you put into making my move a smooth and stress-free experience. Your dedication and professionalism were truly impressive, and I can't imagine how much you must have put in to ensure a successful move.\n\nSecondly, I want to express

**GRPO model (step 400), zero-shot**

| ID | Category | Content | Clean | Note |
| --- | --- | --- | --- | --- |
| w1 | writing | ✗ | ✗ | 4-sentence email (rule: 2); stopped this time |
| w2 | writing | ✗ | ✗ | Far over ~80 words, hit the limit |
| w3 | writing | ✓ | ✗ | One tip, then added a second ("Additionally…") |
| w4 | writing | ✓ | ✓ | |
| f1 | format | ✓ | ✓ | |
| f2 | format | ✓ | ✗ | Correct 7 days, then kept counting to 31 |
| f3 | format | ✗ | ✗ | `#### yes` |
| f4 | format | ✓ | ✗ | Appended "You are an AI assistant…" |
| t1 | translation | ✓ | ✓ | |
| t2 | translation | ✗ | ✗ | Empty `####` line after the translation |
| k1 | facts | ✓ | ✓ | |
| k2 | facts | ✗ | ✗ | `#### Jupiter` |
| k3 | facts | ✓ | ✓ | |
| k4 | facts | ✗ | ✗ | `#### Au` |
| r1 | rewrite | ✗ | ✗ | Repeated the summary after `####` until the limit |
| r2 | rewrite | ✗ | ✗ | Copied the input |
| r3 | rewrite | ✓ | ✗ | Appended "You are an AI assistant…" |
| j1 | judgment | ✗ | ✗ | `#### negative` |
| j2 | judgment | ✗ | ✗ | Wrong (apple), then `#### banana, carrot, grape` |
| j3 | judgment | ✓ | ✗ | Appended "You are an AI assistant…" |
| **Total** | | **10/20** | **5/20** | |

Against SFT (8/20 content, 3/20 clean), nothing was lost. w3 now gives one tip instead of a list, t1 stops cleanly, and k1 answers without a `####` line. The `####` leak into non-math answers, SFT's cost, still affects 7 prompts (k1 no longer, j2 now). On 20 prompts a 2-point gain is within noise: GRPO added no further general cost.

## Stage 3 Summary

Validation subset, 499 problems (±2.2 pts).

| Model | Prompt | Accuracy | No-answer | General: content | General: clean |
| --- | --- | --- | --- | --- | --- |
| Base | zero-shot | 0.2% (48.3% lenient) | 95.2% | 16/20 | 6/20 |
| Base | 8-shot | 46.9% | 4.6% | - | - |
| Base + SFT | zero-shot | 46.5% | 1.2% | 8/20 | 3/20 |
| Base + SFT + GRPO (step 400) | zero-shot | 64.7% | 0.0% | 10/20 | 5/20 |
| Instruct | chat, zero-shot | 53.3% (61.5% lenient) | 22.4% | 17/20 | 16/20 |

GRPO added real math ability. Starting from the SFT model at 46.5%, 400 steps of GRPO reach 64.7% on validation, +18.2 points (paired `z` = 7.2), with a clean format (no-answer 0.0%). That is clearly above Qwen's instruct model as scored (53.3%), and close to it once its format is fixed (61.5%). SFT taught the format by imitating reference solutions; GRPO, rewarded only for correct answers, taught the model to solve problems it couldn't before.

It added no further general cost: 10/20 content and 5/20 clean, against SFT's 8/20 and 3/20, the same within noise. The `####` leak into non-math answers, SFT's cost, remains.

On the unseen test set ([`04_test_eval.ipynb`](04_test_eval.ipynb)) the gain is smaller but holds: SFT 35.0% to GRPO 47.7% (paired `z` = 9.0), level with the instruct model once its format is fixed (46.9%). Validation scores turned out inflated for every model, most likely because GSM8K train problems were in Qwen2.5's training data.